# RAPIDS cuDF, bad data, and a data flywheel on a Kaggle T4

Question: how much does RAPIDS cuDF speed up data preparation, what does bad data cost, and does a data flywheel improve the application while a general benchmark stays blind to it?

Setup: Kaggle GPU T4 x2, internet on. Part 1 uses GPU 0 through `cudf.pandas`. Parts 2 to 5 run on the CPU with scikit-learn (TF-IDF plus logistic regression), so the effect of the data is easy to see. Data: a synthetic inference-request log (part 1) and the AG News dataset (parts 2 to 5). All randomness uses seed 42.

Exam objectives: data.3 (RAPIDS, ETL), data.5 (PII handling), data.7 (evaluation metrics, app-specific evals), data.8 (missing values, duplication, staleness), data.9 (data flywheel, feedback, its challenges), data.11 (data readiness and model performance), data.12 (production quality monitoring).

Terms used below:

- **RAPIDS**: NVIDIA's open-source libraries for data work on GPUs. **cuDF** is its DataFrame library.
- **cudf.pandas**: a mode that runs unchanged pandas code on the GPU. A call that cuDF does not support falls back to CPU pandas.
- **ETL** (extract, transform, load): the steps that move raw data into a form a model or a report can use.
- **PII** (personally identifiable information): data that points to a person, such as an email address or a phone number. **Redaction** replaces it with a placeholder.
- **Exact duplicate**: two rows with the same text. **Near duplicate**: two rows with almost the same text.
- **Stale row**: a row that is too old to describe the world now.
- **Data readiness gate**: a set of checks with limits. The data passes only if every check stays inside its limit.
- **Precision and recall**: precision is the share of flagged rows that are truly faulty. Recall is the share of faulty rows that the check flags.
- **Data leakage**: the same example sits in both the training set and the test set. The test score is then too high.
- **Data flywheel**: a loop. The app serves users, logs what happens, collects feedback, finds failures, labels a few of them, retrains, and serves again.
- **General benchmark**: a fixed public test set. **App-specific eval set**: a held-out test set that comes from your own production traffic.
- **Pseudo-label**: a label that the model itself predicted, with no human check.
- **PSI** (population stability index): a number that measures how far one distribution moved from a reference. A common rule: above 0.2 means a large shift.
- **Delayed labels**: true labels that arrive some time after the prediction. Accuracy from them lags behind.

## Predict before running

Write your own guess first. Then compare it with the tables at the end.

| Step | What you measure | Expected direction |
|---|---|---|
| cuDF at 10 million rows | Speed-up per step, GPU vs CPU, same code | Which steps gain most? Think about parallel work vs sequential work. |
| cuDF at 100 thousand rows | Same speed-ups | Does the GPU still win when the data is small? |
| Quality checks | Precision and recall per fault type | Which fault is easy to find with a rule? Which one needs a model? |
| Readiness gate | Pass or fail before and after cleaning | Faulty data should fail. Cleaned data should pass. |
| Faulty training data | Test accuracy of clean vs faulty vs cleaned | Which fault costs the most accuracy? Does cleaning get it all back? |
| Duplicate leakage | Test accuracy with train rows copied into the test set | The score rises with no real gain. |
| Flywheel, 3 turns | Accuracy on the general benchmark and on the app-specific eval | Which of the two moves? Do they move together? |
| Ablation (a): random sampling | Same label budget, random rows | Failure mining should use the budget better. Check by how much. |
| Ablation (b): pseudo-labels | Model labels its own failures | Does the loop learn anything? |
| Monitoring | Batches until each signal alerts | Which signal alerts first? Which one lags? |

In [ ]:
# 1. Environment. Find cudf. If it is missing, install the wheel that matches the CUDA 13 driver.
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv
import subprocess, sys
def cudf_version():
    r = subprocess.run([sys.executable, "-c", "import cudf; print(cudf.__version__)"], capture_output=True, text=True, timeout=300)
    return (r.stdout.strip() if r.returncode == 0 else None), r.stderr[-300:]
CUDF_VERSION, err = cudf_version()
for pkg in ("cudf-cu13", "cudf-cu12"):
    if CUDF_VERSION:
        break
    print(f"cudf not found. Trying pip install {pkg} ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--extra-index-url=https://pypi.nvidia.com", pkg],
                       capture_output=True, text=True, timeout=1200)
    print("   pip exit code", r.returncode, r.stderr[-200:] if r.returncode else "")
    CUDF_VERSION, err = cudf_version()
HAS_CUDF = CUDF_VERSION is not None
if HAS_CUDF:
    print("cudf", CUDF_VERSION, "is ready")
else:
    print("NOTE: cudf cannot be imported here. Part 1 will time the CPU run only. Parts 2 to 5 do not need cudf.\n", err)

In [ ]:
# 2. Helpers A: faults, checks, readiness gate, PII redaction, model. Pure functions. No GPU, no dataset.
import json, os, re, subprocess, sys, time, zlib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

SEED = 42
LABELS = ["World", "Sports", "Business", "Sci/Tech"]
REF_DATE = pd.Timestamp("2026-11-01")                     # "today" for the staleness check
FAULTS = ["missing", "exact_dup", "near_dup", "stale", "pii", "wrong_label"]
RATES = dict(missing=0.03, exact_dup=0.05, near_dup=0.05, stale=0.05, pii=0.04, wrong_label=0.15)   # injected rates
MARGIN = dict(missing=0.005, exact_dup=0.005, near_dup=0.01, stale=0.01, pii=0.002, wrong_label=0.02)  # gate: allowed rate above the clean reference
NAMES = ["alex.test", "sam.demo", "pat.fake", "kim.sample"]
EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
PHONE = re.compile(r"\b\d{3}[-. ]\d{3}[-. ]\d{4}\b")
T0, RESULTS = time.time(), []
def lap(name): print(f"[{(time.time() - T0) / 60:5.1f} min] {name}")
def add(section, setting, metric, value, note=""):
    RESULTS.append(dict(section=section, setting=setting, metric=metric, value=round(float(value), 4), note=note))

def norm(s): return " ".join(re.sub(r"[^a-z0-9 ]", " ", s.lower()).split())
def redact(s): return PHONE.sub("[PHONE]", EMAIL.sub("[EMAIL]", s))
def make_model(): return make_pipeline(TfidfVectorizer(sublinear_tf=True, min_df=2), LogisticRegression(C=10, max_iter=300))

def near_variant(text, rng):
    w = norm(text).split()                                # lower case, no punctuation, one word dropped
    if len(w) > 6: w.pop(int(rng.integers(len(w))))
    return " ".join(w)

def inject_faults(df, rates=RATES, seed=SEED):
    """df has text, label (0..3), date. Returns a frame with one bool truth column per fault. Copies go at the end."""
    rng = np.random.default_rng(seed)
    d = df[["text", "label", "date"]].reset_index(drop=True)
    n = len(d)
    for f in FAULTS: d[f] = False
    order, pos = rng.permutation(n), 0
    def take(name):                                       # each fault gets its own rows, so the truth is clear
        nonlocal pos
        idx = order[pos: pos + round(rates[name] * n)]; pos += len(idx)
        d.loc[idx, name] = True
        return idx
    d.loc[take("missing"), "text"] = None
    i = take("wrong_label"); d.loc[i, "label"] = (d.label[i] + rng.integers(1, 4, len(i))) % 4
    i = take("stale"); d.loc[i, "date"] = REF_DATE - pd.to_timedelta(rng.integers(900, 1500, len(i)), unit="D")
    i = take("pii"); d.loc[i, "text"] = [f"{t} Contact {a}@example.com or call 555-010-{b:04d}."
                                         for t, a, b in zip(d.text[i], rng.choice(NAMES, len(i)), rng.integers(0, 10000, len(i)))]
    clean_idx = np.flatnonzero(~d[FAULTS].any(axis=1).to_numpy())
    copies = []
    for name in ("exact_dup", "near_dup"):                # copies of rows that carry no other fault
        c = d.loc[rng.choice(clean_idx, round(rates[name] * n), replace=False)].copy()
        c[FAULTS] = False; c[name] = True
        if name == "near_dup": c["text"] = [near_variant(t, rng) for t in c.text]
        copies.append(c)
    return pd.concat([d] + copies, ignore_index=True)

def check_missing(d): return d.text.isna() | d.text.fillna("").str.strip().eq("")
def check_exact_dup(d): return d.text.notna() & d.duplicated("text", keep="first")
def check_stale(d, days=365): return (REF_DATE - d.date).dt.days > days
def check_pii(d): return d.text.fillna("").map(lambda s: bool(EMAIL.search(s) or PHONE.search(s)))

def check_near_dup(d, thr=0.85, chunk=500):
    """Flag a row if an EARLIER row has TF-IDF cosine similarity >= thr. Sparse matrix product, in chunks."""
    X = TfidfVectorizer(stop_words="english", min_df=2).fit_transform(d.text.fillna("").map(norm)).tocsr()
    XT, flag = X.T.tocsr(), np.zeros(len(d), bool)
    for a in range(0, len(d), chunk):
        S = (X[a:a + chunk] @ XT).tocoo()
        keep = (S.data >= thr) & (S.col < S.row + a)
        flag[np.unique(S.row[keep] + a)] = True
    return flag

def check_label_noise(d, thr=0.15, folds=3):
    """Out-of-fold model. Flag a row if the model gives its own label a probability below thr."""
    ok = ~check_missing(d).to_numpy()
    y, flag = d.label.to_numpy(), np.zeros(len(d), bool)
    p = cross_val_predict(make_model(), d.text[ok], y[ok], method="predict_proba",
                          cv=StratifiedKFold(folds, shuffle=True, random_state=SEED))
    flag[ok] = p[np.arange(ok.sum()), y[ok]] < thr
    return flag

def run_checks(d):
    f = pd.DataFrame({"missing": check_missing(d), "exact_dup": check_exact_dup(d),
                      "stale": check_stale(d), "pii": check_pii(d)})
    f["near_dup"] = check_near_dup(d) & ~f.exact_dup.to_numpy()
    f["wrong_label"] = check_label_noise(d)
    return f[FAULTS].astype(bool)

def score_checks(flags, truth):
    rows = []
    for c in FAULTS:
        f, t = flags[c].to_numpy(bool), truth[c].to_numpy(bool)
        tp, fp, fn = (f & t).sum(), (f & ~t).sum(), (~f & t).sum()
        rows.append(dict(check=c, injected=int(t.sum()), flagged=int(f.sum()),
                         precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1)))
    return pd.DataFrame(rows)

def readiness_gate(flags, ref_rates):
    """Pass only if every fault rate is within MARGIN of the rate seen on trusted data."""
    t = pd.DataFrame({"rate": flags.mean(), "limit": pd.Series(ref_rates) + pd.Series(MARGIN)})
    t["ok"] = t.rate <= t.limit
    return bool(t.ok.all()), t

def clean_data(d, flags):
    keep = ~flags[["missing", "exact_dup", "near_dup", "stale", "wrong_label"]].any(axis=1)
    out = d[keep].copy()
    out["text"] = out.text.map(redact)
    return out.reset_index(drop=True)

def leak_mask(train_text, test_text):
    seen = {norm(t) for t in train_text}
    return np.array([norm(t) in seen for t in test_text])

In [ ]:
# 3. Helpers B: flywheel, simulated feedback, labels, drift measure, alert rule. Pure functions.
MIX_OLD, MIX_NEW = [0.4, 0.3, 0.2, 0.1], [0.1, 0.2, 0.3, 0.4]    # topic mix: old training data vs production
HUMAN_NOISE = 0.05                                               # share of wrong human labels

def sample_mix(y, mix, n, rng, free):
    """Draw n row ids with the topic mix. free marks unused rows and is updated, so pools never overlap."""
    y, out = np.asarray(y), []
    for c, p in enumerate(mix):
        ids = rng.choice(np.flatnonzero(free & (y == c)), round(n * p), replace=False)
        free[ids] = False; out.append(ids)
    return rng.permutation(np.concatenate(out))

def restyle(text):
    """New wording style, deterministic. 40 percent of the words with 4 or more letters lose their inner vowels."""
    def f(m):
        w = m.group(0)
        return re.sub(r"(?<=\w)[aeiou]", "", w) if len(w) > 3 and zlib.crc32(w.encode()) % 5 < 2 else w
    return re.sub(r"[a-z]+", f, text.lower())

def noisy_label(y, rate, rng, k=4):
    y = np.asarray(y).copy(); flip = rng.random(len(y)) < rate
    y[flip] = (y[flip] + rng.integers(1, k, flip.sum())) % k
    return y

def simulate_thumbs_down(wrong, rng, p_wrong=0.5, p_right=0.03):
    """Half of the wrong answers get a thumbs down. A few right answers get one too (noise)."""
    u = rng.random(len(wrong))
    return np.where(wrong, u < p_wrong, u < p_right)

def mine_failures(conf, thumbs_down, k, conf_cut=0.6):
    """Failure = thumbs down or confidence below conf_cut. Take the k worst: thumbs down first, then lowest confidence."""
    cand = np.flatnonzero((conf < conf_cut) | thumbs_down)
    score = thumbs_down[cand] + (1 - conf[cand])
    return cand[np.argsort(-score, kind="stable")[:k]]

def run_flywheel(arm, model0, v1, stream, evals, k=250, seed=SEED):
    """arm: mined = failure mining + human labels | random = random rows + human labels | pseudo = failure mining + the model's own labels."""
    rng = np.random.default_rng(seed)
    tx, ty, model = list(v1[0]), list(v1[1]), model0
    def scores(m): return {n: m.score(x, y) for n, (x, y) in evals.items()}
    log = [dict(arm=arm, turn=0, **scores(model))]
    for t, (bx, by) in enumerate(stream, 1):
        p = model.predict_proba(bx); pred = model.classes_[p.argmax(1)]; conf = p.max(1)     # serve and log
        down = simulate_thumbs_down(pred != by, rng)                                          # feedback
        pick = rng.choice(len(bx), k, replace=False) if arm == "random" else mine_failures(conf, down, k)
        new_y = pred[pick] if arm == "pseudo" else noisy_label(by[pick], HUMAN_NOISE, rng)    # label
        tx += [bx[i] for i in pick]; ty += new_y.tolist()
        assert not set(evals["app"][0]) & set(tx), "app eval rows leaked into training"
        model = make_model().fit(tx, ty)                                                      # retrain
        log.append(dict(arm=arm, turn=t, **scores(model), batch_acc=(pred == by).mean(), mean_conf=conf.mean(),
                        down_rate=down.mean(), picked_wrong=(pred[pick] != by[pick]).mean(), label_noise=(new_y != by[pick]).mean()))
    return log, tx

def psi(ref, cur, k=4, eps=1e-4):
    r = np.bincount(ref, minlength=k) / len(ref) + eps
    c = np.bincount(cur, minlength=k) / len(cur) + eps
    return float(((c - r) * np.log(c / r)).sum())

def first_alert(breach, k=2):
    """Index where the signal has breached k times in a row. None if it never does."""
    run = 0
    for i, b in enumerate(breach):
        run = run + 1 if b else 0
        if run >= k: return i

def monitor(model, batches, ref_n=5, lag=3, win=3, seed=SEED):
    """Per-batch signals. Accuracy uses delayed labels: at batch b it sees only batches up to b - lag, over a window of win batches."""
    rng = np.random.default_rng(seed); ok, preds, rows = [], [], []
    for b, (x, y) in enumerate(batches):
        p = model.predict_proba(x); pred = model.classes_[p.argmax(1)]
        ok.append(pred == np.asarray(y)); preds.append(pred)
        late = ok[max(0, b - lag - win + 1): b - lag + 1] if b >= lag else []
        rows.append(dict(batch=b, acc_batch=ok[-1].mean(), conf=p.max(1).mean(), down=simulate_thumbs_down(~ok[-1], rng).mean(),
                         acc_roll=np.concatenate(late).mean() if late else np.nan))
    ref = np.concatenate(preds[:ref_n])
    sig = pd.DataFrame(rows); sig["psi"] = [psi(ref, p) for p in preds]
    return sig

def alert_table(sig, start, ref_n=5, k=2):
    """Alert rule: a signal alerts when it breaches its limit k batches in a row. Limits come from the first ref_n healthy batches."""
    ref = sig.iloc[:ref_n]
    breach = {"accuracy (delayed labels)": sig.acc_roll < ref.acc_batch.mean() - 0.05,
              "mean confidence": sig.conf < ref.conf.mean() - 0.04,
              "thumbs-down rate": sig.down > ref.down.mean() + 0.03,
              "PSI of predicted class mix": sig.psi > 0.2}
    rows = []
    for name, b in breach.items():
        b = b.to_numpy(bool); i = first_alert(b[start:], k)
        rows.append(dict(signal=name, false_alarm_before_drift=first_alert(b[:start], k) is not None,
                         batches_to_alert=None if i is None else i + 1))     # 1 = alert on the first drifted batch
    return pd.DataFrame(rows)

## Part 1: cuDF pandas accelerator

The next cell writes one plain pandas script, `bench.py`. It builds a synthetic request log (timestamps, user ids, model names, latency, token counts, short prompts, some nulls) and times six steps one by one: group-by aggregation, join, sort, filter plus derived columns, string search, and rolling mean plus quantile. Data generation is outside the timings. Each step first runs once on 10,000 rows, outside the timings, because the first GPU calls compile kernels.

Then the script runs twice as a subprocess, with the same code: `python bench.py` (CPU pandas) and `python -m cudf.pandas bench.py` (GPU). Each step returns a checksum, so you can see that both runs compute the same answer. The run repeats at 100 thousand rows, to show where the GPU does not help. A last run uses the `cudf.pandas` profiler to list calls that fell back to the CPU.

In [ ]:
# 4. Write bench.py (plain pandas) and the subprocess runner
BENCH_PATH = "/kaggle/working/bench.py"
BENCH = r"""import sys, time, json
import numpy as np, pandas as pd

N = int(sys.argv[1])
rng = np.random.default_rng(42)

# ---- data generation (NOT timed) ----
MODELS = np.array(["m-small", "m-medium", "m-large", "m-xl", "m-vision", "m-code", "m-embed", "m-audio"])
WORDS = ["summarize", "translate", "classify", "explain", "debug", "extract", "rewrite", "timeout error in"]
THINGS = ["invoice", "report", "email", "ticket", "log", "contract", "review", "chat"]
PROMPTS = np.array([f"{w} {t}" for w in WORDS for t in THINGS])
n_users = max(1000, N // 100)
df = pd.DataFrame({
    "request_id": np.arange(N),
    "ts": pd.Timestamp("2026-01-01") + pd.to_timedelta(np.sort(rng.integers(0, 90 * 86400, N)), unit="s"),
    "user_id": rng.integers(0, n_users, N),
    "model": MODELS[rng.integers(0, len(MODELS), N)],
    "latency_ms": rng.lognormal(5.5, 0.8, N).round(1),
    "input_tokens": rng.integers(10, 4000, N),
    "output_tokens": rng.integers(1, 1500, N),
    "prompt": PROMPTS[rng.integers(0, len(PROMPTS), N)],
})
df.loc[rng.random(N) < 0.02, "latency_ms"] = np.nan
df.loc[rng.random(N) < 0.01, "model"] = None
df.loc[rng.random(N) < 0.03, "prompt"] = None
users = pd.DataFrame({"user_id": np.arange(n_users),
                      "tier": rng.choice(["free", "pro", "team"], n_users),
                      "country": rng.choice(["US", "DE", "MX", "IN", "BR", "FR"], n_users)})

# ---- steps: each returns a small number so the work must finish ----
def groupby_agg(d, u):
    g = d.groupby("user_id").agg(lat_mean=("latency_ms", "mean"), lat_max=("latency_ms", "max"),
                                 tokens=("output_tokens", "sum"), n=("request_id", "count"))
    return float(g.lat_mean.sum() + g.tokens.sum() + g.n.sum())

def join(d, u):
    m = d.merge(u, on="user_id", how="left")
    return float((m.tier == "pro").sum() + len(m))

def sort(d, u):
    return float(d.sort_values(["model", "latency_ms"]).latency_ms.iloc[0])

def filter_derive(d, u):
    x = d.assign(tok_per_s=d.output_tokens / (d.latency_ms / 1000),
                 cost=d.input_tokens * 1e-6 + d.output_tokens * 3e-6, hour=d.ts.dt.hour)
    x = x[x.latency_ms > 500]
    return float(x.tok_per_s.sum() + x.cost.sum() + x.hour.sum())

def string_contains(d, u):
    return float(d.prompt.str.contains("error|timeout", regex=True).sum() + d.prompt.str.contains("invoice", regex=False).sum())

def rolling_quantile(d, u):
    r = d.latency_ms.rolling(1000, min_periods=100).mean()
    q = d.groupby("model").latency_ms.quantile(0.99)
    return float(r.max() + q.sum())

STEPS = [groupby_agg, join, sort, filter_derive, string_contains, rolling_quantile]
warm = df.head(10_000)                       # warm-up run: first GPU calls compile kernels. Not timed.
for f in STEPS:
    f(warm, users)
out = {}
for f in STEPS:
    t = time.perf_counter()
    chk = f(df, users)
    out[f.__name__] = {"s": time.perf_counter() - t, "chk": chk}
print("RESULT " + json.dumps({"n": N, "steps": out}))
"""
Path(BENCH_PATH).write_text(BENCH)

def run_bench(n, gpu, timeout, profile=False):
    cmd = [sys.executable] + (["-m", "cudf.pandas"] + (["--profile"] if profile else []) if gpu else []) + [BENCH_PATH, str(n)]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))
    except subprocess.TimeoutExpired:
        print(f"TIMEOUT after {timeout} s:", " ".join(cmd[1:])); return None, ""
    line = [l for l in r.stdout.splitlines() if l.startswith("RESULT ")]
    if r.returncode or not line:
        print("FAILED:", " ".join(cmd[1:]), "\n--- stderr tail ---\n" + r.stderr[-1500:]); return None, r.stdout
    return json.loads(line[0][7:]), r.stdout

In [ ]:
# 5. Run CPU and GPU at 100 thousand and 10 million rows. Report speed-up per step.
rows = []
for n, limit in ((100_000, 300), (10_000_000, 900)):
    cpu, _ = run_bench(n, False, limit)
    gpu, _ = run_bench(n, True, limit) if HAS_CUDF else (None, "")
    for step, v in (cpu or {"steps": {}})["steps"].items():
        g = gpu["steps"][step] if gpu else None
        rows.append(dict(rows=n, step=step, cpu_s=v["s"], gpu_s=g["s"] if g else np.nan,
                         speedup=v["s"] / g["s"] if g else np.nan,
                         same_result=bool(np.isclose(v["chk"], g["chk"], rtol=1e-6)) if g else None))
sp = pd.DataFrame(rows)
if len(sp):
    tot = sp.groupby("rows")[["cpu_s", "gpu_s"]].sum()
    tot["speedup"] = tot.cpu_s / tot.gpu_s
    print(sp.round(3).to_string(index=False)); print("\nTotal over the six steps:\n", tot.round(3))
    for r in sp.itertuples():
        add("cudf.pandas", f"{r.rows:,} rows | {r.step}", "speedup_x", r.speedup, f"cpu {r.cpu_s:.3f} s, gpu {r.gpu_s:.3f} s")
    for n, r in tot.iterrows():
        add("cudf.pandas", f"{n:,} rows | total", "speedup_x", r.speedup, f"cpu {r.cpu_s:.2f} s, gpu {r.gpu_s:.2f} s")
if HAS_CUDF:
    prof, out = run_bench(100_000, True, 300, profile=True)           # which calls ran on the GPU, which on the CPU
    print("\ncudf.pandas profiler output (tail):\n" + "\n".join(out.splitlines()[-40:]))
lap("part 1 done")

## Part 2: data quality checks and a readiness gate

Data: AG News, 12,000 training rows (4 topics). The text is the news text. Each row gets a random date in the last 120 days before 2026-11-01.

The next cells inject six faults at known rates, with the truth stored in one column per fault:

| Fault | How it is injected |
|---|---|
| missing | the text is set to null |
| exact_dup | a copy of a clean row is added at the end |
| near_dup | a copy with lower case, no punctuation and one word dropped is added at the end |
| stale | the date moves back 900 to 1,500 days |
| pii | a fake email (`name@example.com`) and a fake phone (`555-010-xxxx`) are added to the text |
| wrong_label | the label changes to another topic |

Each fault uses its own rows, so the truth is clear. The checks never see the truth columns. Then each check is scored against the truth with precision and recall.

The checks: null or empty text; `duplicated` on the text; TF-IDF cosine similarity of at least 0.85 to an earlier row; date older than 365 days; regular expressions for email and phone; and an out-of-fold model that flags a row when it gives the row's own label a probability below 0.15.

Real AG News also has faults that you did not inject, for example real near-duplicate wire stories. A check can flag them. They count as false positives here, so precision can look lower than the true quality of the check.

The readiness gate compares each flagged rate with the rate on the untouched sample plus a margin. The gate fails if any check is above its limit.

In [ ]:
# 6. Load AG News, build the clean sample, and measure the natural fault rates on it
from datasets import load_dataset
ds = load_dataset("fancyzhx/ag_news")               # the bare name "ag_news" fails on current datasets
tr, te = ds["train"].to_pandas(), ds["test"].to_pandas()
assert ds["train"].features["label"].names == LABELS
rng = np.random.default_rng(SEED)
base = tr.sample(12_000, random_state=SEED).reset_index(drop=True)
base["date"] = REF_DATE - pd.to_timedelta(rng.integers(0, 120, len(base)), unit="D")
test_honest = te.sample(4_000, random_state=SEED)   # the test set for parts 2 and 3. It never has a fault.
ref_flags = run_checks(base)
ref_rates = ref_flags.mean()
print("Share of rows flagged on the untouched AG News sample (natural rate, the gate reference):")
print(ref_rates.round(4).to_string())
lap("clean sample checked")

In [ ]:
# 7. Inject the faults, run the checks, score them against the truth, run the gate
faulty = inject_faults(base)
print("rows before / after injection:", len(base), "/", len(faulty))
flags = run_checks(faulty)                           # the checks see text, label and date only
sc = score_checks(flags, faulty)
print(sc.round(3).to_string(index=False))
ok, gate_tab = readiness_gate(flags, ref_rates)
print("\nREADINESS GATE on faulty data:", "PASS" if ok else "FAIL"); print(gate_tab.round(4).to_string())
for r in sc.itertuples():
    add("data quality", r.check, "precision", r.precision, f"{r.flagged} flagged, {r.injected} injected")
    add("data quality", r.check, "recall", r.recall)
add("data quality", "faulty data", "gate_pass", ok)
lap("faulty data checked")

In [ ]:
# 8. Clean the data: drop flagged rows, redact PII. Then run the same checks and the same gate again.
cleaned = clean_data(faulty, flags)
flags2 = run_checks(cleaned)
ok2, gate_tab2 = readiness_gate(flags2, ref_rates)
print(f"rows: faulty {len(faulty)} -> cleaned {len(cleaned)}")
print("READINESS GATE on cleaned data:", "PASS" if ok2 else "FAIL"); print(gate_tab2.round(4).to_string())
redacted = faulty.loc[faulty.pii, "text"].map(redact)
print("\nPII rows injected:", int(faulty.pii.sum()), "| still matching the PII patterns after redaction:", int(redacted.map(lambda s: bool(EMAIL.search(s) or PHONE.search(s))).sum()))
print("example before:", faulty.loc[faulty.pii, "text"].iloc[0][-90:])
print("example after: ", redacted.iloc[0][-90:])
add("data quality", "cleaned data", "gate_pass", ok2)
add("data quality", "PII redaction", "rows_still_matching", redacted.map(lambda s: bool(EMAIL.search(s) or PHONE.search(s))).sum())
lap("part 2 done")

## Part 3: what bad data costs

The same model trains three times: TF-IDF plus logistic regression. Only the training data changes. The test set is the same 4,000 untouched AG News rows. Training data A is the clean sample. B is the faulty data. C is the faulty data after the checks cleaned it. A missing text becomes an empty string for B.

Then a leakage test. The model from A scores two test sets of 2,000 rows each. One set is honest. The other set has 1,000 honest rows and 1,000 rows copied from the training set. `leak_mask` finds the copies, and removing them gives back an honest score.

Stale rows do not change the score on a fixed test set. They matter when the world moves. Part 4 makes the world move.

In [ ]:
# 9. Test accuracy: clean vs faulty vs cleaned training data
Xt, yt = test_honest.text.tolist(), test_honest.label.to_numpy()
models = {}
for name, d in (("A clean sample", base), ("B faulty data", faulty), ("C faulty data after cleaning", cleaned)):
    models[name] = make_model().fit(d.text.fillna("").tolist(), d.label)
    a = models[name].score(Xt, yt)
    print(f"{name:30s} train rows {len(d):6d} | test accuracy {a:.4f}")
    add("effect on accuracy", name, "test_accuracy", a, f"{len(d)} training rows")
lap("accuracy done")

In [ ]:
# 10. Duplicate leakage: train rows copied into the test set
m = models["A clean sample"]
honest = test_honest.iloc[:2000]
leaky = pd.concat([test_honest.iloc[2000:3000], base.sample(1000, random_state=1)[["text", "label"]]])
tt = base.text.tolist()
for name, t in (("honest test set (2000)", honest), ("leaky test set (1000 honest + 1000 copied)", leaky)):
    a, lk = m.score(t.text.tolist(), t.label), leak_mask(tt, t.text).mean()
    print(f"{name:45s} accuracy {a:.4f} | share of test rows found in train {lk:.3f}")
    add("effect on accuracy", name, "test_accuracy", a, f"leak share {lk:.3f}")
no_leak = leaky[~leak_mask(tt, leaky.text)]
a = m.score(no_leak.text.tolist(), no_leak.label)
print(f"{'leaky set after removing the copies':45s} accuracy {a:.4f} on {len(no_leak)} rows")
add("effect on accuracy", "leaky set after removing copies", "test_accuracy", a, f"{len(no_leak)} rows")
lap("part 3 done")

## Part 4: the data flywheel, three turns

**Setup.**

- The **general benchmark** is 2,000 rows of the standard AG News test split, all four topics in equal share, original wording.
- **v1** trains on 1,000 old rows with a skewed topic mix (World 40%, Sports 30%, Business 20%, Sci/Tech 10%) and the original wording.
- The **production stream** drifts in two ways. The topic mix changes (Business 30%, Sci/Tech 40%). The wording changes with `restyle`, a deterministic transform: 40% of the longer words lose their inner vowels (for example, "company" becomes "cmpny").
- The **app-specific eval set** has 1,500 rows from the same production source. It is held out. No eval row is served, and none is used for training. The notebook asserts this.

**One turn.** Serve a batch of 2,500 rows. Log the prediction and the confidence. Simulate user feedback: half of the wrong answers get a thumbs down, and 3% of the right answers get one by mistake. A failure is a thumbs down or a confidence below 0.6. Take the 250 worst failures (thumbs down first, then lowest confidence). A simulated human labels them, with 5% wrong labels. Add them to the training data. Retrain from scratch. Score both eval sets.

**Ablations.** Each one changes one thing and keeps the same stream and the same budget of 250 labels per turn.

- (a) `random`: pick 250 random rows instead of failures.
- (b) `pseudo`: pick the same failures, but label them with the model's own predictions, with no human check. This is the feedback loop.

The table marks a turn `general_blind` when the app score rose by more than 1 point while the general score rose by less than 0.5 point.

In [ ]:
# 11. Build the pools. Pools never share rows. Check that the app eval set never overlaps the training data.
tr_text, y_tr = tr.text.tolist(), tr.label.to_numpy()
free, rng = np.ones(len(tr), bool), np.random.default_rng(SEED)
v1_idx = sample_mix(y_tr, MIX_OLD, 1000, rng, free)
stream_idx = [sample_mix(y_tr, MIX_NEW, 2500, rng, free) for _ in range(3)]
app_idx = sample_mix(y_tr, MIX_NEW, 1500, rng, free)
mon_old = [sample_mix(y_tr, MIX_OLD, 300, rng, free) for _ in range(24)]      # used in part 5
mon_new = [sample_mix(y_tr, MIX_NEW, 300, rng, free) for _ in range(24)]
gen_idx = sample_mix(te.label.to_numpy(), [0.25] * 4, 2000, rng, np.ones(len(te), bool))

v1 = ([tr_text[i] for i in v1_idx], y_tr[v1_idx])
stream = [([restyle(tr_text[i]) for i in ix], y_tr[ix]) for ix in stream_idx]
seen = set(v1[0]) | {t for x, _ in stream for t in x}
app = [(restyle(tr_text[i]), y_tr[i]) for i in app_idx]
app = [(t, y) for t, y in app if t not in seen]                                # drop rows whose text was served or trained on
evals = {"general": (te.text.iloc[gen_idx].tolist(), te.label.to_numpy()[gen_idx]),
         "app": ([t for t, _ in app], np.array([y for _, y in app]))}
assert not set(evals["app"][0]) & seen, "app eval overlaps the v1 data or the served stream"
assert not set(app_idx) & (set(v1_idx) | {i for ix in stream_idx for i in ix}), "app eval ids overlap"
model_v1 = make_model().fit(*v1)
print(f"v1 train {len(v1[0])} | stream 3 x {len(stream[0][0])} | app eval {len(app)} | general benchmark {len(evals['general'][0])}")
print("example, original :", tr_text[stream_idx[0][0]][:110]); print("example, restyled :", stream[0][0][0][:110])
lap("pools built")

In [ ]:
# 12. Run the three flywheel arms and compare the two eval sets turn by turn
logs = []
for arm in ("mined", "random", "pseudo"):
    log, tx = run_flywheel(arm, model_v1, v1, stream, evals)
    assert not set(evals["app"][0]) & set(tx)                                  # the app eval set never entered training
    logs += log
fw = pd.DataFrame(logs)
fw["d_general"], fw["d_app"] = fw.groupby("arm").general.diff(), fw.groupby("arm").app.diff()
fw["general_blind"] = (fw.d_app > 0.01) & (fw.d_general < 0.005)
print(fw[["arm", "turn", "general", "app", "d_general", "d_app", "general_blind"]].round(4).to_string(index=False))
print("\nTelemetry of the served batch (before the retrain of that turn):")
print(fw[fw.turn > 0][["arm", "turn", "batch_acc", "mean_conf", "down_rate", "picked_wrong", "label_noise"]].round(3).to_string(index=False))
for r in fw.itertuples():
    add("flywheel", f"{r.arm} | turn {r.turn}", "general_accuracy", r.general); add("flywheel", f"{r.arm} | turn {r.turn}", "app_accuracy", r.app)
lap("part 4 done")

## Part 5: production quality monitoring

The deployed model is v1, and nobody retrains it here. Traffic comes in 24 batches of 300 rows. Batches 0 to 7 look like the training data. From batch 8 the share of drifted rows (new topic mix and new wording) grows by 20 points per batch and reaches 100% at batch 12.

Four signals are computed per batch:

- **Accuracy from delayed labels.** True labels arrive 3 batches late. At batch b the monitor sees only batches up to b - 3, averaged over 3 batches.
- **Mean confidence** of the predictions. It needs no label.
- **Thumbs-down rate.** It needs user feedback (same simulation as part 4).
- **PSI of the predicted class mix** against the first 5 batches. It needs no label.

**Alert rule:** a signal alerts when it breaches its limit in 2 batches in a row. The limits come from the first 5 healthy batches: accuracy 5 points below, confidence 0.04 below, thumbs-down rate 3 points above, PSI above 0.2. The table shows how many drifted batches each signal needed before its alert, and whether it alerted before the drift (a false alarm).

In [ ]:
# 13. Monitor the deployed v1 model on a stream that drifts from batch 8
START = 8
batches = []
for b in range(24):
    n_new = round(min(1.0, max(0, b - START + 1) / 5) * 300)
    old_ids, new_ids = mon_old[b][:300 - n_new], mon_new[b][:n_new]
    batches.append(([tr_text[i] for i in old_ids] + [restyle(tr_text[i]) for i in new_ids], np.concatenate([y_tr[old_ids], y_tr[new_ids]])))
sig = monitor(model_v1, batches)
print(sig.round(3).to_string(index=False))
at = alert_table(sig, START)
print("\nAlert table (batches_to_alert: 1 = alert at the first drifted batch):"); print(at.to_string(index=False))
for r in at.itertuples():
    add("monitoring", r.signal, "batches_to_alert", np.nan if r.batches_to_alert is None else r.batches_to_alert, f"false alarm before drift: {r.false_alarm_before_drift}")
lap("part 5 done")

In [ ]:
# 14. Summary table, also written to /kaggle/working/data_flywheel_results.csv
df = pd.DataFrame(RESULTS)
df.to_csv("/kaggle/working/data_flywheel_results.csv", index=False)
with pd.option_context("display.max_rows", 500, "display.width", 200, "display.max_colwidth", 70):
    print(df.to_string(index=False))
print(f"\ntotal wall time {(time.time() - T0) / 60:.1f} min | saved /kaggle/working/data_flywheel_results.csv")

## What to look for

- **cuDF speed-ups.** Compare the speed-up of each step at 10 million rows with the same step at 100 thousand rows. Check `same_result`: both runs must give the same checksum. Read the profiler tail: it lists the calls that ran on the CPU. A GPU run also pays fixed costs, such as launching kernels and moving data. Small tables may not repay them.
- **Check scores.** Compare precision and recall per fault. Rules (null, exact match, date, regular expression) and model-based checks (near duplicates, wrong labels) can behave very differently. Remember that some flagged rows are real faults you did not inject.
- **Gate.** Compare the gate on faulty and cleaned data. A gate has no meaning without limits. Here the limits come from a trusted reference plus a margin.
- **Accuracy.** Compare A, B and C. Then compare the honest test set with the leaky one. A high score on a leaky set tells you nothing about new data.
- **Flywheel.** Compare `general` and `app` after each turn, and read the `general_blind` column. Then compare the three arms: the telemetry columns `picked_wrong` and `label_noise` explain the gaps. The simulated feedback is a model of users, not real users. The numbers show the mechanism, not a forecast.
- **Monitoring.** Compare the batches each signal needed. Signals without labels can alert earlier. Signals with labels are closer to the truth but arrive late. A signal with a false alarm before the drift has limits that are too tight for its noise.
- This run uses one seed. A difference of a few tenths of a point between arms can be noise.

## Exam rules this proves

1. **data.3**: `cudf.pandas` runs unchanged pandas code on the GPU and falls back to the CPU for unsupported calls. Measure the speed-up per step and per data size, and check that the results match.
2. **data.5**: Find PII with rules and redact it before data goes to training or logs. Prove it by running the same detector on the output.
3. **data.8, data.11**: Measure missing values, duplicates, staleness and wrong labels with checks, and block the data with a readiness gate. Faulty data lowers model quality. Duplicates across train and test raise the score falsely.
4. **data.7, data.9**: A general benchmark measures the old distribution. An app-specific eval set, held out from production traffic, measures the app. A flywheel loop is serve, log, collect feedback, mine failures, label, retrain. Its challenges are a fixed label budget, noisy feedback, noisy labels, eval leakage, and a feedback loop when the model labels its own output.
5. **data.12**: Monitor accuracy from delayed labels, confidence, user feedback and drift (PSI) per batch. Set alert rules with limits from healthy data. Signals without labels are fast. Signals with labels are slow but direct.